# 03. Đánh giá trên test và phân tích lỗi

Chỉ chạy notebook này sau khi đã chốt model ở notebook 02. Nếu thư mục `EVAL_DIR` đã có thì notebook chỉ đọc lại kết quả cũ chứ không chấm lại, để tập test không bị dùng nhiều lần.

Phần cuối tổng hợp các lượt thử trực tiếp trong GUI (chế độ thử nghiệm) từ `results/measurements.csv`.

In [ ]:
# Mở notebook từ thư mục notebooks/ hay thư mục gốc đều chạy được
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Audio, Image, Markdown, display
pd.set_option('display.max_colwidth', 80)
print('Thư mục làm việc:', ROOT)

In [ ]:
# Tham số
SPLIT    = os.environ.get('VOVC_SPLIT', 'data/manifest_split.csv')
MODEL    = os.environ.get('VOVC_MODEL', 'models/nb_model_01.joblib')
EVAL_DIR = os.environ.get('VOVC_EVAL', 'results/eval_nb_01')
REQUIRE_NEW_SPEAKER = False      # True khi đã có giọng người thứ hai trong test
TRIALS   = os.environ.get('VOVC_TRIALS', 'results/measurements.csv')
EVENTS   = os.environ.get('VOVC_EVENTS', 'results/events.jsonl')

## 1. Chấm trên test (một lần)

In [ ]:
import json
from ml.evaluate import evaluate
from ml.report import summary_md

metrics_path = Path(EVAL_DIR) / 'metrics.json'
if metrics_path.exists():
    print(f'Đọc lại kết quả đã có trong {EVAL_DIR}.')
    metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
else:
    metrics = evaluate(SPLIT, MODEL, EVAL_DIR, require_new_speaker=REQUIRE_NEW_SPEAKER)
display(Markdown(summary_md(metrics)))

In [ ]:
from ml.report import report

if not (Path(EVAL_DIR) / 'confusion_matrix.png').exists():
    report(EVAL_DIR)
display(Image(filename=str(Path(EVAL_DIR) / 'confusion_matrix.png'), width=560))
display(Image(filename=str(Path(EVAL_DIR) / 'per_class_f1.png'), width=560))

## 2. Nghe các mẫu bị đoán sai

In [ ]:
pred = pd.read_csv(Path(EVAL_DIR) / 'predictions.csv')
pred['file'] = [str((Path(SPLIT).parent / p).resolve()) for p in pred.path]
wrong = pred[pred.predicted != pred.expected]
print(f'Sai {len(wrong)}/{len(pred)}')
wrong[['path', 'expected', 'predicted', 'raw_label', 'score']]

In [ ]:
for _, r in wrong.head(8).iterrows():
    display(Markdown(f"`{r.path}`: thật **{r.expected}**, đoán **{r.predicted}** (score {r.score:.2f})"))
    if Path(r.file).exists():
        display(Audio(filename=r.file))

## 3. Score của mẫu đúng và mẫu sai

Nếu phần lớn mẫu sai có score thấp hơn ngưỡng thì nâng ngưỡng lên sẽ biến "làm sai lệnh" thành "không làm gì", an toàn hơn khi điều khiển thiết bị. Còn nếu mẫu sai có score cao thì model đang tự tin mà sai, cần thêm dữ liệu hoặc đặc trưng tốt hơn.

In [ ]:
import numpy as np
fig, ax = plt.subplots(figsize=(6.5, 3.5))
bins = np.linspace(0, 1, 21)
ax.hist(pred.score[pred.predicted == pred.expected], bins=bins, alpha=0.7, label='đúng', color='#54A24B')
ax.hist(pred.score[pred.predicted != pred.expected], bins=bins, alpha=0.7, label='sai', color='#E45756')
ax.axvline(metrics['threshold'], color='black', ls='--', label=f"ngưỡng {metrics['threshold']:.2f}")
ax.set_xlabel('score'); ax.set_ylabel('số mẫu'); ax.legend(); ax.set_title('Phân bố score trên test')
plt.show()

## 4. Lượt thử trực tiếp và độ trễ

In [ ]:
from ml.trial_report import summarize_trials
from ml import viz

if Path(TRIALS).exists() and summarize_trials(TRIALS):
    display(pd.DataFrame(summarize_trials(TRIALS)).T.round(3))
else:
    print('Chưa có lượt thử. Trong GUI tick «Ghi lượt thử» rồi nói vài lệnh.')

latency = []
if Path(EVENTS).exists():
    for line in Path(EVENTS).read_text(encoding='utf-8').splitlines():
        if line.strip():
            e = json.loads(line)
            if e.get('source') == 'voice' and 'post_record_ms' in e:
                latency.append(e['post_record_ms'])
if latency:
    viz.latency_hist(latency, f'Độ trễ sau khi thu xong ({len(latency)} lượt giọng nói)')
    plt.show()
else:
    print('Chưa có lượt giọng nói trong event log.')

## Khi ghi vào báo cáo

- Ghi rõ `evaluation_type`. Nếu là `same_speaker_new_session` thì kết quả chưa chứng minh được là nhận được giọng người lạ.
- Ghi kèm số mẫu, người nói, micro, điều kiện thu. Độ trễ ghi median và p95.
- Trước khi commit nhớ Clear All Outputs.